# A1组 6.2 实验：PINN 正问题与反问题

> 本 Notebook 只包含 6.2 节，配套正文为 `A1-G12-正文.md`。

- **代码载体：** 全部代码在本 Notebook 内，按 `ch06-fill-011` → `ch06-fill-012` → `ch06-fill-013` 顺序执行。
- **相对路径：** 从本文件所在的提交包根目录启动 Jupyter；图片写入 `图片/`，数据写入 `数据/`，不新建其他文件夹。
- **环境：** Python 3.12.15 / NumPy 2.5.4 / Matplotlib 3.11.2；正文表 6.2-1、表 6.2-2 与 Notebook 输出均由本环境 25 次完整运行得到，见文末“6.2 实际运行记录”。
- **依赖：** 仅 NumPy、Matplotlib；不导入 scipy/torch。
- **数据与图：** 25 次逐次结果、损失/预测数据、三张图分别写入 `数据/` 与 `图片/`。


## 6.2 PINN：正问题与反问题（2学时）

**本节目标：** 能区分求解未知函数和识别未知参数，能写出含待估参数与观测项的 PINN 损失，能说明观测数量、位置、噪声和先验如何影响可辨识性，并用合成观测完成一个最小反问题实验。

### 概念与推导（B 版合并）

6.1 的正问题是方程、边界和源项已知，只求 $u$。6.2 只把常数系数改成未知：

$$
-\kappa u''(x)=\pi^2\sin(\pi x),\qquad x\in(0,1),\qquad u(0)=u(1)=0,
$$

其中 $\kappa>0$ 待估，精确解族为

$$
u_\kappa(x)=\frac1\kappa\sin(\pi x).
$$

$\kappa$ 只改变整体幅度 $1/\kappa$，不改变形状；零边界对一切 $\kappa$ 都成立，所以边界值本身不含 $\kappa$ 的信息。

**边界零观测为什么失败。** 观测算子 $H(\kappa)=(u_\kappa(x_1),\dots,u_\kappa(x_{N_d}))$ 在仅边界观测时为 $H_b(\kappa)=(0,0)$，不是单射；损失在 $\kappa$ 方向恒为零，Fisher 信息 $I(\kappa)=0$。这不是采样不足，而是模型层不可辨识；加密配点、换优化器都不会有帮助。若网络只能近似表示 $u_\kappa$，则平坦方向是近似平坦，参数漂移由逼近误差和优化路径决定。

**内点观测为什么有效。** 对 $x_d\in(0,1)$，记 $s_d=\sin(\pi x_d)$，则

$$
u_\kappa(x_d)=\frac{s_d}{\kappa},\qquad
J_d=\frac{\partial u_\kappa}{\partial\kappa}=-\frac{s_d}{\kappa^2}.
$$

$|J_d|$ 在 $x=1/2$ 最大，近边界按 $x$ 线性衰减。高斯观测 $y_d=u_\kappa(x_d)+\varepsilon_d$ 下，

$$
I(\kappa)=\frac{1}{\sigma^2\kappa^4}\sum_d\sin^2(\pi x_d),\qquad
\mathrm{sd}(\hat\kappa)\ge\frac{\sigma\kappa^2}{\sqrt{\sum_d\sin^2(\pi x_d)}}.
$$

这是 CR 基准，不是 PINN 的误差保证；PINN 有逼近误差和优化偏差，实测散度可能更大。

**PINN 损失。** 本节使用

$$
\mathcal L=w_r\mathcal L_r+w_b\mathcal L_b+w_d\mathcal L_d+w_p\mathcal R_p,
$$

$$
\mathcal L_r=\frac1{N_r}\sum_i(-\kappa_\alpha u_\theta''(x_i)-f(x_i))^2,\quad
\mathcal L_b=\frac1{N_b}\sum_j(u_\theta(x_j^b)-g_D(x_j^b))^2,
$$

$$
\mathcal L_d=\frac1{N_d}\sum_d(u_\theta(x_d)-y_d)^2,\qquad
\kappa_\alpha=e^\alpha.
$$

主实验取 $w_r=w_b=w_d=1$、$w_p=0$、软边界。$\kappa=e^\alpha$ 保证正性；链式法则给 $\partial\mathcal L/\partial\alpha=\kappa\,\partial\mathcal L/\partial\kappa$。在子族 $u_\theta=a\sin(\pi x)$ 中，残差项只依赖乘积 $\kappa a$，数据项只依赖幅度 $a$；所以残差定形状、数据定尺度，二者联合才定 $\kappa$。

**误差指标。** 本模型有精确关系

$$
\frac{\|u_{\hat\kappa}-u_{\kappa^\star}\|_2}{\|u_{\kappa^\star}\|_2}
=\frac{|\Delta\kappa|}{\hat\kappa},
$$

残差与观测 RMSE 也都是 $|\Delta\kappa|$ 的单调函数。这是单模式特例，一般 PDE 不成立；报告时不能用一个指标替代另一个。

**限制。** 本节只处理一维、线性、常数 $\kappa$、零 Dirichlet、合成观测；不处理空间变系数、非线性、多维、系统偏差、真实数据和其他网络结构。先验/正则化和硬约束只作为机制讨论，不用于主实验。

### 实验设计：先预测，再运行

| 要填写的项目 | 本组内容 |
| --- | --- |
| 要回答的问题与运行前预测 | 内点观测能否恢复 $\kappa$？仅边界观测是否出现“损失低、参数错”？观测位置和噪声如何改变结果？预测：E1/E2 参数误差约 5%–10%；E3 明显变差；E4/E5 接近平坦、参数误差 30% 左右；E1/E2 的解 $L^2$ 应小于 E3/E4/E5。 |
| 输入、参数、单位、数据来源与划分 | 方程 $-\kappa u''=\pi^2\sin(\pi x)$，$x\in[0,1]$（无量纲），$\kappa$ 无量纲，真值 $\kappa^\star=2$。合成观测 $y_d=u^\star(x_d)+\varepsilon_d$，$\varepsilon_d\sim N(0,\sigma^2)$，$\sigma\in\{0,0.005\}$。E1/E2 内点 $1/6,\dots,5/6$；E3 近边界 $0.01,\dots,0.05$；E4/E5 边界 $0,1$。$N_r=256$ 配点，$N_b=2$ 边界点；评价网格 1001 个单元中心，避开观测点。训练观测与评价网格分开；5 个种子 2026–2030。 |
| 独立参考解/基线及选择理由 | 解析解 $u^\star=\sin(\pi x)/2$，可直接代入、无离散误差；6.1 的软/硬约束与有限差分结果作为正问题基线（$1.096\times10^{-5}$、$3.790\times10^{-6}$、$1.285\times10^{-6}$）；CR 下界作为统计基准。 |
| 误差指标、容差及其依据 | 参数相对误差 $|\hat\kappa-\kappa^\star|/\kappa^\star$；解相对 $L^2$；观测 RMSE；独立残差 RMS；边界误差。工作容差：E1/E2 参数误差不超过 10%、解 $L^2$ 不超过 10%；E3–E5 参数误差预期超过 20%、解 $L^2$ 超过 30%；E4/E5 必须呈平坦。依据是 CR 预测（E2 相对标准差约 0.58%，E4/E5 的 $S=0$）与 6.1 正问题基线；CR 只是基准，不是 PINN 的精度保证。 |
| 环境、种子、设备与运行预算 | Python 3.12.15 / NumPy 2.5.4 / Matplotlib 3.11.2；CPU；5 个种子 × 5 个设置 = 25 次；每次 2000 步 Adam，lr $0.01$；多数单次 2.6–4.4 s，E5 一个种子受系统负载影响 10.9 s，总墙钟 96.9 s。 |

**运行要求。** 实际运行 `ch06-fill-012`，保存全部 25 次结果；`ch06-fill-013` 同时展示 5 个种子，不只展示最好一次。结果段只报告本次 Python 复现结果。


In [1]:
# 6.2 输入与独立参考
# 运行顺序：本 cell -> ch06-fill-012 -> ch06-fill-013。
# 依赖：仅 numpy + matplotlib（不导入 scipy/torch）。
import csv
import json
import sys
import time
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

# 固定种子：模板全局种子为 2026；5 次独立初始化依次为 2026–2030。
rng = np.random.default_rng(2026)
SEEDS = [2026, 2027, 2028, 2029, 2030]

KAPPA_TRUE = 2.0
N_R = 256
N_EVAL = 1001
EPOCHS = 2000
LR = 0.01

# 主评价网格用单元中心，避开 x=0,1 以及 0.01,...,0.05 等观测点。
X_EVAL = (np.arange(N_EVAL) + 0.5) / N_EVAL

SETTINGS = [
    {
        "id": "E1",
        "name": "内部均匀, 无噪声",
        "x_d": np.array([1 / 6, 2 / 6, 3 / 6, 4 / 6, 5 / 6], dtype=float),
        "sigma": 0.0,
    },
    {
        "id": "E2",
        "name": "内部均匀, 有噪声",
        "x_d": np.array([1 / 6, 2 / 6, 3 / 6, 4 / 6, 5 / 6], dtype=float),
        "sigma": 0.005,
    },
    {
        "id": "E3",
        "name": "近边界, 无噪声",
        "x_d": np.array([0.01, 0.02, 0.03, 0.04, 0.05], dtype=float),
        "sigma": 0.0,
    },
    {
        "id": "E4",
        "name": "仅边界, 无噪声",
        "x_d": np.array([0.0, 1.0], dtype=float),
        "sigma": 0.0,
    },
    {
        "id": "E5",
        "name": "仅边界, 有噪声",
        "x_d": np.array([0.0, 1.0], dtype=float),
        "sigma": 0.005,
    },
]

WORK = Path.cwd()
OUT = WORK / "数据"
FIG = WORK / "图片"
assert OUT.is_dir() and FIG.is_dir(), "缺少 数据/ 或 图片/ 目录；请在包含这两个目录的提交包根目录下启动 Jupyter"

# 独立参考：解析解族 u_kappa = sin(pi x)/kappa。
X_TRUE = np.sin(np.pi * X_EVAL) / KAPPA_TRUE

# CR 理论预测（仅作基准，不是 PINN 的误差保证）。
# 布局 A: x_d=1/6..5/6, S^2=3；布局 B: 0.01..0.05, S^2≈0.0540；布局 C: S=0。
S2_A = 3.0
S2_B = float(np.sum(np.sin(np.pi * SETTINGS[2]["x_d"]) ** 2))
CR = {
    "E1": {"S2": S2_A, "sigma": 0.0, "sd": 0.0},
    "E2": {"S2": S2_A, "sigma": 0.005, "sd": 0.005 * KAPPA_TRUE**2 / np.sqrt(S2_A)},
    "E3": {"S2": S2_B, "sigma": 0.0, "sd": 0.0},
    "E4": {"S2": 0.0, "sigma": 0.0, "sd": np.inf},
    "E5": {"S2": 0.0, "sigma": 0.005, "sd": np.inf},
}

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__, "| Matplotlib:", matplotlib.__version__)
print("设置: kappa*=2, 网络 1-32-32-1 tanh, Adam 2000 步, lr=0.01, soft BC")
print("种子:", SEEDS)
print("\n观测设置与独立参考：")
print("| id | x_d | N_d | sigma | 参考解 |")
print("|---|---|---:|---:|---|")
for s in SETTINGS:
    print(
        f"| {s['id']} | {np.round(s['x_d'], 4).tolist()} | {len(s['x_d'])} | "
        f"{s['sigma']} | u*=sin(pi x)/2 |"
    )

print("\nCR 标准差预测（无偏、线性化、已知 sigma）：")
for sid, v in CR.items():
    if np.isinf(v["sd"]):
        print(f"  {sid}: S^2={v['S2']:.4g}, sd=+inf（不可辨识）")
    else:
        rel = v["sd"] / KAPPA_TRUE
        print(f"  {sid}: S^2={v['S2']:.4g}, sd={v['sd']:.6g}, 相对={rel:.3%}")

# 观测样例（每个设置只展示前 3 个点；实际训练用全部点）。
print("\n观测样例（clean / noisy）：")
for s in SETTINGS:
    x = s["x_d"]
    y_clean = np.sin(np.pi * x) / KAPPA_TRUE
    noise_rng = np.random.default_rng(SEEDS[0])
    y = y_clean + s["sigma"] * noise_rng.normal(size=x.shape)
    print(f"  {s['id']}: " + ", ".join(
        f"x={xx:.4f}, y={yy:.6f}" for xx, yy in zip(x[:3], y[:3])
    ))


Python: 3.12.15
NumPy: 2.5.4 | Matplotlib: 3.11.2
设置: kappa*=2, 网络 1-32-32-1 tanh, Adam 2000 步, lr=0.01, soft BC
种子: [2026, 2027, 2028, 2029, 2030]

观测设置与独立参考：
| id | x_d | N_d | sigma | 参考解 |
|---|---|---:|---:|---|
| E1 | [0.1667, 0.3333, 0.5, 0.6667, 0.8333] | 5 | 0.0 | u*=sin(pi x)/2 |
| E2 | [0.1667, 0.3333, 0.5, 0.6667, 0.8333] | 5 | 0.005 | u*=sin(pi x)/2 |
| E3 | [0.01, 0.02, 0.03, 0.04, 0.05] | 5 | 0.0 | u*=sin(pi x)/2 |
| E4 | [0.0, 1.0] | 2 | 0.0 | u*=sin(pi x)/2 |
| E5 | [0.0, 1.0] | 2 | 0.005 | u*=sin(pi x)/2 |

CR 标准差预测（无偏、线性化、已知 sigma）：
  E1: S^2=3, sd=0, 相对=0.000%
  E2: S^2=3, sd=0.011547, 相对=0.577%
  E3: S^2=0.05397, sd=0, 相对=0.000%
  E4: S^2=0, sd=+inf（不可辨识）
  E5: S^2=0, sd=+inf（不可辨识）

观测样例（clean / noisy）：
  E1: x=0.1667, y=0.250000, x=0.3333, y=0.433013, x=0.5000, y=0.500000
  E2: x=0.1667, y=0.246034, x=0.3333, y=0.434216, x=0.5000, y=0.490518
  E3: x=0.0100, y=0.015705, x=0.0200, y=0.031395, x=0.0300, y=0.047054
  E4: x=0.0000, y=0.000000, x=1.0000, y=0.000000
  E5

In [2]:
# 6.2 方法实现与运行
# 纯 NumPy：1-32-32-1 tanh，解析二阶导，手写反向传播，Adam 2000 步。
# 运行顺序：先执行 ch06-fill-011（配置），再执行本 cell。
if "SETTINGS" not in globals():
    raise RuntimeError("请先执行 ch06-fill-011（6.2 输入与独立参考）。")


class MLP:
    """1-32-32-1 tanh MLP，返回 u, u_x, u_xx。"""

    def __init__(self, seed, sizes=(1, 32, 32, 1)):
        rng_local = np.random.default_rng(seed)
        self.sizes = sizes
        self.W = []
        self.b = []
        for fan_in, fan_out in zip(sizes[:-1], sizes[1:]):
            limit = np.sqrt(6.0 / (fan_in + fan_out))  # Glorot uniform
            self.W.append(rng_local.uniform(-limit, limit, size=(fan_out, fan_in)))
            self.b.append(np.zeros(fan_out, dtype=float))

    def forward(self, x):
        x = np.asarray(x, dtype=float).reshape(-1, 1)
        h, h1, h2 = x, np.ones_like(x), np.zeros_like(x)
        cache = []
        for i in range(len(self.W) - 1):
            W, b = self.W[i], self.b[i]
            z = h @ W.T + b
            z1 = h1 @ W.T
            z2 = h2 @ W.T
            a = np.tanh(z)
            s = 1.0 - a * a
            a1 = s * z1
            a2 = -2.0 * a * s * z1 * z1 + s * z2
            cache.append((h, h1, h2, z, a, s, z1, z2))
            h, h1, h2 = a, a1, a2
        Wo, bo = self.W[-1], self.b[-1]
        u = h @ Wo.T + bo
        u1 = h1 @ Wo.T
        u2 = h2 @ Wo.T
        cache.append((h, h1, h2, u, u1, u2))
        return u, u1, u2, cache

    def backward(self, cache, gu, gu1, gu2):
        grads_W = [None] * len(self.W)
        grads_b = [None] * len(self.b)
        h, h1, h2, u, u1, u2 = cache[-1]
        Wo = self.W[-1]
        grads_W[-1] = gu.T @ h + gu1.T @ h1 + gu2.T @ h2
        # 输出层偏置只进入 u，不进入 u_x/u_xx
        grads_b[-1] = gu.sum(0)
        gh = gu @ Wo
        gh1 = gu1 @ Wo
        gh2 = gu2 @ Wo
        for i in range(len(self.W) - 2, -1, -1):
            h_prev, h1_prev, h2_prev, z, a, s, z1, z2 = cache[i]
            # a = tanh(z), a1=s*z1, a2=-2*a*s*z1^2+s*z2
            gz = (
                gh * s
                + gh1 * (-2.0 * a * s * z1)
                + gh2 * (-2.0 * s * (1.0 - 3.0 * a * a) * z1 * z1 - 2.0 * a * s * z2)
            )
            gz1 = gh1 * s + gh2 * (-4.0 * a * s * z1)
            gz2 = gh2 * s
            W = self.W[i]
            grads_W[i] = gz.T @ h_prev + gz1.T @ h1_prev + gz2.T @ h2_prev
            # 隐藏层偏置只进入 z，不进入 z_x/z_xx
            grads_b[i] = gz.sum(0)
            gh = gz @ W
            gh1 = gz1 @ W
            gh2 = gz2 @ W
        return grads_W, grads_b


def loss_and_grads(net, alpha, x_r, x_b, x_d, y_d):
    kappa = float(np.exp(alpha[0]))

    # PDE residual
    u_r, u1_r, u2_r, cache_r = net.forward(x_r)
    f_r = np.pi**2 * np.sin(np.pi * x_r)
    r = -kappa * u2_r - f_r
    L_r = float(np.mean(r * r))
    gu_r = np.zeros_like(u_r)
    gu1_r = np.zeros_like(u_r)
    gu2_r = (2.0 / r.size) * r * (-kappa)
    dW_r, db_r = net.backward(cache_r, gu_r, gu1_r, gu2_r)
    dkappa = float(np.mean(2.0 * r * (-u2_r)))

    # soft boundary
    u_b, _, _, cache_b = net.forward(x_b)
    L_b = float(np.mean(u_b * u_b))
    gu_b = (2.0 / u_b.size) * u_b
    gu1_b = np.zeros_like(u_b)
    gu2_b = np.zeros_like(u_b)
    dW_b, db_b = net.backward(cache_b, gu_b, gu1_b, gu2_b)

    # data
    u_d, _, _, cache_d = net.forward(x_d)
    L_d = float(np.mean((u_d - y_d) ** 2))
    gu_d = (2.0 / u_d.size) * (u_d - y_d)
    gu1_d = np.zeros_like(u_d)
    gu2_d = np.zeros_like(u_d)
    dW_d, db_d = net.backward(cache_d, gu_d, gu1_d, gu2_d)

    dW = [a + b + c for a, b, c in zip(dW_r, dW_b, dW_d)]
    db = [a + b + c for a, b, c in zip(db_r, db_b, db_d)]
    dalpha = np.array([dkappa * kappa], dtype=float)
    loss = L_r + L_b + L_d
    return loss, (dW, db, dalpha), (L_r, L_b, L_d)


def total_loss(net, alpha, x_r, x_b, x_d, y_d):
    kappa = float(np.exp(alpha[0]))
    u_r, _, u2_r, _ = net.forward(x_r)
    f_r = np.pi**2 * np.sin(np.pi * x_r)
    L_r = float(np.mean((-kappa * u2_r - f_r) ** 2))
    u_b, _, _, _ = net.forward(x_b)
    L_b = float(np.mean(u_b * u_b))
    u_d, _, _, _ = net.forward(x_d)
    L_d = float(np.mean((u_d - y_d) ** 2))
    return L_r + L_b + L_d


def init_adam(params):
    return [np.zeros_like(p, dtype=float) for p in params], [
        np.zeros_like(p, dtype=float) for p in params
    ]


def adam_step(params, grads, m, v, t, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
    for i in range(len(params)):
        g = grads[i]
        m[i] = beta1 * m[i] + (1.0 - beta1) * g
        v[i] = beta2 * v[i] + (1.0 - beta2) * (g * g)
        mhat = m[i] / (1.0 - beta1**t)
        vhat = v[i] / (1.0 - beta2**t)
        params[i] -= lr * mhat / (np.sqrt(vhat) + eps)


def make_data(setting, seed):
    rng_local = np.random.default_rng(seed)
    x_d = setting["x_d"].reshape(-1, 1)
    y_clean = np.sin(np.pi * x_d) / KAPPA_TRUE
    y_d = y_clean + setting["sigma"] * rng_local.normal(size=x_d.shape)
    x_r = np.linspace(0.0, 1.0, N_R).reshape(-1, 1)
    x_b = np.array([[0.0], [1.0]])
    return x_r, x_b, x_d, y_d


def evaluate(net, alpha, x_d, y_d):
    kappa = float(np.exp(alpha[0]))
    u, u1, u2, _ = net.forward(X_EVAL.reshape(-1, 1))
    u_true = np.sin(np.pi * X_EVAL) / KAPPA_TRUE
    rel_l2 = float(np.linalg.norm(u.ravel() - u_true) / np.linalg.norm(u_true))
    max_err = float(np.max(np.abs(u.ravel() - u_true)))
    residual = -kappa * u2.ravel() - np.pi**2 * np.sin(np.pi * X_EVAL)
    residual_rms = float(np.sqrt(np.mean(residual**2)))

    u_d, _, _, _ = net.forward(x_d)
    obs_rmse = float(np.sqrt(np.mean((u_d - y_d) ** 2)))
    u_b, _, _, _ = net.forward(np.array([[0.0], [1.0]]))
    boundary_err = float(np.max(np.abs(u_b)))
    return {
        "kappa_hat": kappa,
        "param_rel_err": abs(kappa - KAPPA_TRUE) / KAPPA_TRUE,
        "rel_l2": rel_l2,
        "max_err": max_err,
        "obs_rmse": obs_rmse,
        "residual_rms": residual_rms,
        "boundary_err": boundary_err,
    }


def run_one(setting, seed, epochs=EPOCHS, lr=LR, verbose=False):
    x_r, x_b, x_d, y_d = make_data(setting, seed)
    net = MLP(seed=seed)
    alpha = np.array([0.0], dtype=float)
    params = net.W + net.b + [alpha]
    m, v = init_adam(params)
    loss_hist = np.empty(epochs)
    kappa_hist = np.empty(epochs)
    Lr_hist = np.empty(epochs)
    Lb_hist = np.empty(epochs)
    Ld_hist = np.empty(epochs)
    t0 = time.perf_counter()
    for epoch in range(1, epochs + 1):
        loss, (dW, db, dalpha), (Lr, Lb, Ld) = loss_and_grads(
            net, alpha, x_r, x_b, x_d, y_d
        )
        adam_step(params, dW + db + [dalpha], m, v, epoch, lr=lr)
        loss_hist[epoch - 1] = loss
        kappa_hist[epoch - 1] = float(np.exp(alpha[0]))
        Lr_hist[epoch - 1] = Lr
        Lb_hist[epoch - 1] = Lb
        Ld_hist[epoch - 1] = Ld
        if verbose and (epoch == 1 or epoch % 400 == 0):
            print(
                f"{setting['id']} seed={seed} epoch={epoch:4d} "
                f"loss={loss:.4e} kappa={np.exp(alpha[0]):.6f}"
            )
    elapsed = time.perf_counter() - t0
    metrics = evaluate(net, alpha, x_d, y_d)
    metrics.update(
        {
            "setting": setting["id"],
            "name": setting["name"],
            "seed": seed,
            "sigma": setting["sigma"],
            "n_d": int(len(setting["x_d"])),
            "epochs": epochs,
            "lr": lr,
            "time_s": elapsed,
            "final_loss": float(loss_hist[-1]),
            "Lr_final": float(Lr_hist[-1]),
            "Lb_final": float(Lb_hist[-1]),
            "Ld_final": float(Ld_hist[-1]),
        }
    )
    hist = {
        "loss": loss_hist,
        "kappa": kappa_hist,
        "Lr": Lr_hist,
        "Lb": Lb_hist,
        "Ld": Ld_hist,
    }
    return metrics, hist, net, alpha, (x_r, x_b, x_d, y_d)


def gradient_check():
    """有限差分核验：u_xx、参数梯度、alpha 梯度。"""
    rng_local = np.random.default_rng(123)
    net = MLP(seed=999)
    alpha = np.array([np.log(1.7)])
    x_r = np.linspace(0.02, 0.98, 7).reshape(-1, 1)
    x_b = np.array([[0.0], [1.0]])
    x_d = np.array([[0.2], [0.4], [0.6]])
    y_d = np.sin(np.pi * x_d) / KAPPA_TRUE
    loss, (dW, db, dalpha), _ = loss_and_grads(net, alpha, x_r, x_b, x_d, y_d)
    params = net.W + net.b + [alpha]
    grads = dW + db + [dalpha]

    x0 = np.array([[0.37]])
    h = 1e-5
    u0, _, _, _ = net.forward(x0)
    up, _, _, _ = net.forward(x0 + h)
    um, _, _, _ = net.forward(x0 - h)
    u2_fd = (up - 2 * u0 + um) / h**2
    _, _, u2_an, _ = net.forward(x0)
    uxx_rel = abs(float(u2_an[0, 0] - u2_fd[0, 0])) / max(
        1.0, abs(float(u2_fd[0, 0]))
    )

    checks = []
    for _ in range(6):
        pi = int(rng_local.integers(0, len(params)))
        idx = tuple(int(rng_local.integers(0, s)) for s in params[pi].shape)
        eps = 1e-6
        old = params[pi][idx]
        params[pi][idx] = old + eps
        lp = total_loss(net, alpha, x_r, x_b, x_d, y_d)
        params[pi][idx] = old - eps
        lm = total_loss(net, alpha, x_r, x_b, x_d, y_d)
        params[pi][idx] = old
        fd = (lp - lm) / (2 * eps)
        an = float(grads[pi][idx])
        checks.append(abs(an - fd) / max(1e-8, abs(fd)))
    print(
        f"gradient check: uxx_rel={uxx_rel:.3e}, "
        f"max_param_rel={max(checks):.3e}, loss={loss:.6e}"
    )
    return uxx_rel, max(checks)


def run_all():
    all_metrics, histories, predictions = [], {}, {}
    total_t0 = time.perf_counter()
    for setting in SETTINGS:
        for seed in SEEDS:
            metrics, hist, net, alpha, _ = run_one(setting, seed=seed, verbose=False)
            all_metrics.append(metrics)
            histories[f"{setting['id']}_{seed}"] = hist
            u_eval, _, _, _ = net.forward(X_EVAL.reshape(-1, 1))
            predictions[f"{setting['id']}_{seed}"] = u_eval.ravel()
            print(
                f"{setting['id']} seed={seed}: "
                f"kappa={metrics['kappa_hat']:.4f}, "
                f"param={metrics['param_rel_err']:.2%}, "
                f"L2={metrics['rel_l2']:.2%}, "
                f"loss={metrics['final_loss']:.2e}, "
                f"time={metrics['time_s']:.1f}s"
            )
    total = time.perf_counter() - total_t0
    with open(OUT / "A1组-6.2-反问题汇总.json", "w", encoding="utf-8") as f:
        json.dump(all_metrics, f, ensure_ascii=False, indent=2)
    with open(OUT / "A1组-6.2-反问题逐次结果.csv", "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(all_metrics[0].keys()))
        writer.writeheader()
        writer.writerows(all_metrics)
    np.savez_compressed(
        OUT / "A1组-6.2-训练历史.npz",
        **{
            f"{run_id}_{key}": arr
            for run_id, hist in histories.items()
            for key, arr in hist.items()
        },
    )
    np.savez_compressed(
        OUT / "A1组-6.2-预测数据.npz",
        x_eval=X_EVAL,
        u_true=np.sin(np.pi * X_EVAL) / KAPPA_TRUE,
        **predictions,
    )
    print(f"总运行时间: {total:.1f}s；结果已写入 数据/")
    return all_metrics, histories, total


def summarize(all_metrics):
    print("\n| 设置 | kappa mean±sd | 参数误差 mean±sd | 解 L2 mean±sd | 观测 RMSE | 残差 RMS | 边界误差 | 时间/s |")
    print("|---|---:|---:|---:|---:|---:|---:|---:|")
    for setting in SETTINGS:
        rows = [r for r in all_metrics if r["setting"] == setting["id"]]
        k = np.array([r["kappa_hat"] for r in rows])
        p = np.array([r["param_rel_err"] for r in rows])
        l2 = np.array([r["rel_l2"] for r in rows])
        obs = np.array([r["obs_rmse"] for r in rows])
        res = np.array([r["residual_rms"] for r in rows])
        bnd = np.array([r["boundary_err"] for r in rows])
        t = np.array([r["time_s"] for r in rows])
        print(
            f"| {setting['id']} | {k.mean():.4f}±{k.std(ddof=1):.4f} | "
            f"{p.mean():.2%}±{p.std(ddof=1):.2%} | "
            f"{l2.mean():.2%}±{l2.std(ddof=1):.2%} | "
            f"{obs.mean():.4f} | {res.mean():.4f} | "
            f"{bnd.mean():.4f} | {t.mean():.1f} |"
        )


gradient_check()
results, histories, total_time = run_all()
summarize(results)


gradient check: uxx_rel=7.235e-07, max_param_rel=9.723e-08, loss=4.107066e+01
E1 seed=2026: kappa=1.8427, param=7.87%, L2=4.89%, loss=1.27e-03, time=2.8s
E1 seed=2027: kappa=1.8591, param=7.04%, L2=8.22%, loss=3.10e-01, time=2.9s
E1 seed=2028: kappa=1.8682, param=6.59%, L2=4.69%, loss=3.46e-02, time=3.4s
E1 seed=2029: kappa=1.8907, param=5.47%, L2=3.29%, loss=3.73e-04, time=4.2s
E1 seed=2030: kappa=1.8500, param=7.50%, L2=4.63%, loss=1.40e-03, time=4.4s
E2 seed=2026: kappa=1.8559, param=7.20%, L2=4.38%, loss=1.12e-03, time=4.1s
E2 seed=2027: kappa=1.8648, param=6.76%, L2=3.98%, loss=9.08e-04, time=3.3s
E2 seed=2028: kappa=1.8865, param=5.68%, L2=4.29%, loss=1.05e-01, time=3.0s
E2 seed=2029: kappa=1.8833, param=5.84%, L2=3.64%, loss=3.92e-04, time=3.3s
E2 seed=2030: kappa=1.8766, param=6.17%, L2=3.86%, loss=1.14e-03, time=4.1s
E3 seed=2026: kappa=1.2643, param=36.79%, L2=55.80%, loss=6.27e-04, time=3.6s
E3 seed=2027: kappa=1.3695, param=31.53%, L2=44.10%, loss=8.22e-04, time=3.0s
E3 see

In [3]:
# 6.2 误差图表与比较
# 读取 ch06-fill-012 写出的 A1组-6.2-反问题汇总.json / A1组-6.2-训练历史.npz / A1组-6.2-预测数据.npz，
# 生成修正版 2×2 主图、E5 收敛图、E1 解对比图，并打印聚合表。
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

WORK = Path.cwd()
OUT = WORK / "数据"
FIG = WORK / "图片"
assert FIG.is_dir(), "缺少 图片/ 目录；请在包含 图片/ 的提交包根目录下启动 Jupyter"

with open(OUT / "A1组-6.2-反问题汇总.json", encoding="utf-8") as f:
    results = json.load(f)
hist = np.load(OUT / "A1组-6.2-训练历史.npz")
pred = np.load(OUT / "A1组-6.2-预测数据.npz")

SETTING_IDS = ["E1", "E2", "E3", "E4", "E5"]
SEEDS = [2026, 2027, 2028, 2029, 2030]
SHORT = {
    "E1": "E1\ninternal\n$\\sigma=0$",
    "E2": "E2\ninternal\n$\\sigma=.005$",
    "E3": "E3\nnear-bound\n$\\sigma=0$",
    "E4": "E4\nboundary\n$\\sigma=0$",
    "E5": "E5\nboundary\n$\\sigma=.005$",
}
plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "legend.fontsize": 7,
        "axes.grid": True,
        "grid.alpha": 0.25,
    }
)


def rows_for(sid):
    return [r for r in results if r["setting"] == sid]


def print_tables():
    print("## 聚合表（5 seeds）")
    print("| 设置 | Python kappa mean±sd | 参数误差 mean±sd | 解 L2 mean±sd | 观测 RMSE | 残差 RMS | 边界误差 | 时间/s |")
    print("|---|---:|---:|---:|---:|---:|---:|---:|")
    for sid in SETTING_IDS:
        rows = rows_for(sid)
        k = np.array([r["kappa_hat"] for r in rows])
        p = np.array([r["param_rel_err"] for r in rows])
        l2 = np.array([r["rel_l2"] for r in rows])
        obs = np.array([r["obs_rmse"] for r in rows])
        res = np.array([r["residual_rms"] for r in rows])
        bnd = np.array([r["boundary_err"] for r in rows])
        t = np.array([r["time_s"] for r in rows])
        print(
            f"| {sid} | {k.mean():.4f}±{k.std(ddof=1):.4f} | "
            f"{p.mean():.2%}±{p.std(ddof=1):.2%} | "
            f"{l2.mean():.2%}±{l2.std(ddof=1):.2%} | "
            f"{obs.mean():.4f} | {res.mean():.4f} | "
            f"{bnd.mean():.4f} | {t.mean():.1f} |"
        )

    print("\n## 种子间分布")
    print("| 设置 | 参数误差 min | 参数误差 mean | 参数误差 max | 参数误差 sd | 解 L2 min | 解 L2 mean | 解 L2 max | 解 L2 sd |")
    print("|---|---:|---:|---:|---:|---:|---:|---:|---:|")
    for sid in SETTING_IDS:
        rows = rows_for(sid)
        p = np.array([r["param_rel_err"] for r in rows]) * 100
        l2 = np.array([r["rel_l2"] for r in rows]) * 100
        print(
            f"| {sid} | {p.min():.2f}% | {p.mean():.2f}% | {p.max():.2f}% | {p.std(ddof=1):.2f}% | "
            f"{l2.min():.2f}% | {l2.mean():.2f}% | {l2.max():.2f}% | {l2.std(ddof=1):.2f}% |"
        )

    print("\n## 逐次运行（不是只展示最好一次）")
    print("| 设置 | seed | kappa | 参数误差 | 解 L2 | 观测 RMSE | 残差 RMS | 边界误差 | 时间/s |")
    print("|---|---:|---:|---:|---:|---:|---:|---:|---:|")
    for sid in SETTING_IDS:
        for r in rows_for(sid):
            print(
                f"| {sid} | {r['seed']} | {r['kappa_hat']:.4f} | "
                f"{r['param_rel_err']:.2%} | {r['rel_l2']:.2%} | "
                f"{r['obs_rmse']:.4f} | {r['residual_rms']:.4f} | "
                f"{r['boundary_err']:.4f} | {r['time_s']:.1f} |"
            )


def plot_main():
    fig, axes = plt.subplots(2, 2, figsize=(11.2, 8.0))
    ax = axes[0, 0]
    for seed in SEEDS:
        ax.plot(hist[f"E1_{seed}_kappa"], lw=0.9, alpha=0.55, label=f"seed {seed}")
    ax.plot(np.mean([hist[f"E1_{s}_kappa"] for s in SEEDS], axis=0), color="k", lw=2.2, label="mean")
    ax.axhline(2.0, color="tab:red", ls="--", lw=1.2, label=r"true $\kappa^\star=2$")
    ax.set_title("E1: parameter convergence (internal, no noise)")
    ax.set_xlabel("epoch")
    ax.set_ylabel(r"$\hat{\kappa}$")
    ax.set_ylim(0.95, 2.05)
    ax.legend(ncol=2, loc="lower right")

    ax = axes[0, 1]
    for seed in SEEDS:
        ax.plot(hist[f"E4_{seed}_kappa"], lw=0.9, alpha=0.55, label=f"seed {seed}")
    ax.plot(np.mean([hist[f"E4_{s}_kappa"] for s in SEEDS], axis=0), color="k", lw=2.2, label="mean")
    ax.axhline(2.0, color="tab:red", ls="--", lw=1.2, label=r"true $\kappa^\star=2$")
    ax.set_title("E4: boundary-only convergence (no noise)")
    ax.set_xlabel("epoch")
    ax.set_ylabel(r"$\hat{\kappa}$")
    ax.set_ylim(0.95, 2.05)

    ax = axes[1, 0]
    x = np.linspace(0, 1, 401)
    ax.plot(x, np.sin(np.pi * x), "k-", lw=1.8, label=r"$|J|/\max|J|=\sin(\pi x)$")
    x_in = np.array([1 / 6, 2 / 6, 3 / 6, 4 / 6, 5 / 6])
    ax.scatter(x_in, np.sin(np.pi * x_in), s=42, color="tab:blue", zorder=3, label="E1 internal obs")
    ax.scatter([0, 1], [0, 0], s=42, color="tab:red", zorder=3, label="E4 boundary obs")
    ax.set_title("Observation sensitivity (normalized)")
    ax.set_xlabel("x")
    ax.set_ylabel(r"$|J|/\max|J|$")
    ax.set_ylim(-0.03, 1.06)
    ax.legend(loc="lower center")

    ax = axes[1, 1]
    means, stds = [], []
    for sid in SETTING_IDS:
        vals = np.array([r["param_rel_err"] for r in rows_for(sid)]) * 100
        means.append(vals.mean())
        stds.append(vals.std(ddof=1))
    xx = np.arange(len(SETTING_IDS))
    ax.bar(xx, means, yerr=stds, capsize=4, color="tab:blue", alpha=0.72, label="Python mean ± sd (5 seeds)")
    for i, sid in enumerate(SETTING_IDS):
        vals = np.array([r["param_rel_err"] for r in rows_for(sid)]) * 100
        ax.scatter(np.full(vals.size, xx[i]) + np.linspace(-0.08, 0.08, vals.size), vals, s=16, color="k", alpha=0.65, zorder=3)
    ax.set_xticks(xx)
    ax.set_xticklabels([SHORT[s] for s in SETTING_IDS])
    ax.set_ylabel(r"$|\hat{\kappa}-\kappa^\star|/\kappa^\star$  (%)")
    ax.set_title("Parameter error across observation settings")
    ax.legend(loc="upper left")

    fig.suptitle("6.2 PINN inverse problem: Python reproduction (5 seeds)", y=0.995)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(FIG / "A1组-6.2-反问题主图.png", bbox_inches="tight")
    plt.show()
    plt.close(fig)


def plot_e5():
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    for seed in SEEDS:
        ax.plot(hist[f"E5_{seed}_kappa"], lw=1.0, alpha=0.6, label=f"seed {seed}")
    ax.plot(np.mean([hist[f"E5_{s}_kappa"] for s in SEEDS], axis=0), color="k", lw=2.2, label="mean")
    ax.axhline(2.0, color="tab:red", ls="--", lw=1.2, label=r"true $\kappa^\star=2$")
    ax.set_title("E5: boundary-only convergence (with noise, 5 seeds)")
    ax.set_xlabel("epoch")
    ax.set_ylabel(r"$\hat{\kappa}$")
    ax.set_ylim(0.95, 2.05)
    ax.legend(ncol=2)
    fig.tight_layout()
    fig.savefig(FIG / "A1组-6.2-E5边界噪声收敛.png", bbox_inches="tight")
    plt.show()
    plt.close(fig)


def plot_solution():
    x = pred["x_eval"]
    u_true = pred["u_true"]
    fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.2))
    ax = axes[0]
    ax.plot(x, u_true, "k-", lw=2.2, label=r"$u^\star=\sin(\pi x)/2$")
    for seed in SEEDS:
        ax.plot(x, pred[f"E1_{seed}"], lw=0.9, alpha=0.6, label=f"seed {seed}")
    ax.set_title("E1: all 5 predictions vs exact solution")
    ax.set_xlabel("x")
    ax.set_ylabel("u")
    ax.legend(ncol=2)
    ax = axes[1]
    for seed in SEEDS:
        ax.semilogy(x, np.abs(pred[f"E1_{seed}"] - u_true), lw=0.9, alpha=0.65, label=f"seed {seed}")
    ax.set_title("E1: pointwise absolute error (all seeds)")
    ax.set_xlabel("x")
    ax.set_ylabel(r"$|u_\theta-u^\star|$")
    ax.legend(ncol=2)
    fig.tight_layout()
    fig.savefig(FIG / "A1组-6.2-E1解对比.png", bbox_inches="tight")
    plt.show()
    plt.close(fig)


print_tables()
plot_main()
plot_e5()
plot_solution()
print("\n图已保存到: 图片/")


## 聚合表（5 seeds）
| 设置 | Python kappa mean±sd | 参数误差 mean±sd | 解 L2 mean±sd | 观测 RMSE | 残差 RMS | 边界误差 | 时间/s |
|---|---:|---:|---:|---:|---:|---:|---:|
| E1 | 1.8621±0.0186 | 6.89%±0.93% | 5.14%±1.83% | 0.0181 | 0.0983 | 0.0252 | 3.5 |
| E2 | 1.8734±0.0128 | 6.33%±0.64% | 4.03%±0.31% | 0.0159 | 0.0543 | 0.0186 | 3.6 |
| E3 | 1.3662±0.0610 | 31.69%±3.05% | 44.82%±6.39% | 0.0130 | 0.0214 | 0.0146 | 3.2 |
| E4 | 1.3395±0.0467 | 33.02%±2.33% | 48.86%±5.62% | 0.0151 | 0.0735 | 0.0192 | 3.5 |
| E5 | 1.3376±0.0476 | 33.12%±2.38% | 50.11%±5.03% | 0.0025 | 0.0224 | 0.0039 | 5.6 |

## 种子间分布
| 设置 | 参数误差 min | 参数误差 mean | 参数误差 max | 参数误差 sd | 解 L2 min | 解 L2 mean | 解 L2 max | 解 L2 sd |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| E1 | 5.47% | 6.89% | 7.87% | 0.93% | 3.29% | 5.14% | 8.22% | 1.83% |
| E2 | 5.68% | 6.33% | 7.20% | 0.64% | 3.64% | 4.03% | 4.38% | 0.31% |
| E3 | 29.08% | 31.69% | 36.79% | 3.05% | 40.27% | 44.82% | 55.80% | 6.39% |
| E4 | 30.57% | 33.02% | 36.61% | 2.33% | 44.01% | 48.

### 结果解释与本节结论

**证据口径。** 本轮 Python 3.12.15 / NumPy 2.5.4 结果来自 `ch06-fill-012` 的 5 个种子（2026–2030）× 5 个设置，共 25 次；表中为均值±标准差。结果段只报告本次 Python 复现，不引用其他实现的数值。

**表 1 · Python 复现（5 seeds）**

| 设置 | 种子数 | $\hat\kappa$ (mean±sd) | 参数误差 (mean±sd) | 解 $L^2$ (mean±sd) | 观测 RMSE | 残差 RMS | 边界误差 | 时间/s |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| E1 内部无噪 | 5 | 1.8621±0.0186 | 6.89%±0.93% | 5.14%±1.83% | 0.0181 | 0.0983 | 0.0252 | 3.5 |
| E2 内部有噪 | 5 | 1.8734±0.0128 | 6.33%±0.64% | 4.03%±0.31% | 0.0159 | 0.0543 | 0.0186 | 3.6 |
| E3 近边界无噪 | 5 | 1.3662±0.0610 | 31.69%±3.05% | 44.82%±6.39% | 0.0130 | 0.0214 | 0.0146 | 3.2 |
| E4 仅边界无噪 | 5 | 1.3395±0.0467 | 33.02%±2.33% | 48.86%±5.62% | 0.0151 | 0.0735 | 0.0192 | 3.5 |
| E5 仅边界有噪 | 5 | 1.3376±0.0476 | 33.12%±2.38% | 50.11%±5.03% | 0.0025 | 0.0224 | 0.0039 | 5.6 |

**表 2 · 种子间分布**

| 设置 | 参数误差 min | 参数误差 mean | 参数误差 max | 参数误差 sd | 解 $L^2$ min | 解 $L^2$ mean | 解 $L^2$ max | 解 $L^2$ sd | $\hat\kappa$ 范围 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| E1 | 5.47% | 6.89% | 7.87% | 0.93% | 3.29% | 5.14% | 8.22% | 1.83% | 1.8427–1.8907 |
| E2 | 5.68% | 6.33% | 7.20% | 0.64% | 3.64% | 4.03% | 4.38% | 0.31% | 1.8559–1.8865 |
| E3 | 29.08% | 31.69% | 36.79% | 3.05% | 40.27% | 44.82% | 55.80% | 6.39% | 1.2643–1.4183 |
| E4 | 30.57% | 33.02% | 36.61% | 2.33% | 44.01% | 48.86% | 57.76% | 5.62% | 1.2677–1.3887 |
| E5 | 30.65% | 33.12% | 36.74% | 2.38% | 44.42% | 50.11% | 57.91% | 5.03% | 1.2651–1.3871 |

逐次运行表见 `ch06-fill-013` 的输出和 `数据/A1组-6.2-反问题逐次结果.csv`；本节没有只挑最好一次。E1 的参数误差范围为 5.47%–7.87%，E2 为 5.68%–7.20%；E4 为 30.57%–36.61%，E5 为 30.65%–36.74%。这些范围说明种子散度是结果的一部分，不能用一个点代替。

![Python 复现主图](图片/A1组-6.2-反问题主图.png)

*图 1 · 左上/E1、右上/E4 的 5 个种子与均值；左下为归一化灵敏度 $|J|/\max|J|=\sin(\pi x)$；右下为参数误差均值±标准差与 5 个种子散点。保留 2×2 布局，补齐 5 个柱标签，把“敏感性”纵轴明确为归一化 $|J|$。*

![E5 收敛](图片/A1组-6.2-E5边界噪声收敛.png)

*图 2 · E5（仅边界、$\sigma=0.005$）的 5 个种子与均值；红线为 $\kappa^\star=2$。*

![E1 解对比](图片/A1组-6.2-E1解对比.png)

*图 3 · E1 的 5 个种子：左为预测与解析解，右为逐点绝对误差（对数纵轴）。不挑选最好种子。*

**结果解释.**

1. **E1/E2 说明内点观测确实提供了 $\kappa$ 的尺度信息。** 参数误差均值 6.89%/6.33%，解 $L^2$ 均值 5.14%/4.03%，与参数误差同量级；这与单模式关系 $e_u=|\Delta\kappa|/\hat\kappa$ 一致。E1 的种子散度仍可见（5.47%–7.87%），所以只报一次会高估方法。
2. **E3 说明“有内点”不等于“信息足够”。** 观测点 $0.01,\dots,0.05$ 的 $s_d$ 只有 0.03–0.16，$J_d\propto s_d$，参数误差 31.69%，与仅边界观测同量级。位置设计比“有没有内点”更细。
3. **E4/E5 说明边界零观测的结构性失败。** 两者参数误差 33.02%/33.12%，解 $L^2$ 48.86%/50.11%；损失可以很低，但参数仍偏离真值。图 1 右上、图 2 显示 $\hat\kappa$ 早期跳到 1.26–1.42 后基本平坦，种子只在平台上漂移。
4. **噪声没有把不可辨识变成可辨识。** E5 比 E4 只多 $\sigma=0.005$ 的边界噪声，参数误差均值只差 0.10 个百分点；噪声既没有提供 $\kappa$ 信息，也没有把估计拉回真值。需要注意 E4/E5 的 $\mathcal L_b$ 与 $\mathcal L_d$ 在 $[0,1]$ 上重合：已知零边界与带噪边界观测相互竞争，E5 的边界误差/观测 RMSE 更小可能是“拟合噪声”，不能解读为“参数更准”。
5. **与 6.1 基线的关系。** 6.1 的正问题软/硬约束分别达到 $1.096\times10^{-5}$ 和 $3.790\times10^{-6}$，说明正问题本身可以被解得很准；6.2 的误差主要不是网络无法表示 $\sin(\pi x)$，而是反问题的可辨识性与观测设计。E1/E2 的 $10^{-2}$ 量级误差、E3–E5 的 $10^{-1}$ 量级误差都应从这个角度读，而不是误以为“网络容量不够”。

**本节结论.** 内点观测是识别 $\kappa$ 的必要条件；仅边界零观测时，$\kappa$ 结构性不可辨识，损失再低也不能证明参数对。内点观测的位置决定信息强弱：近边界内点仍可能病态。反问题必须同时报告参数误差、解误差、观测设置和多种子结果；只报总损失或最好一次会误导。

### 易错点与讨论

**（1）损失低不等于参数对。** 双参数模型 $-\kappa u''=A\pi^2\sin(\pi x)$ 中，$(A,\kappa)=(1,2)$ 与 $(2,4)$ 给出逐点相同的解 $u=\sin(\pi x)$，残差、边界值和内点观测都相同；只观测 $u$ 时无法区分，Fisher 矩阵秩 1。E4/E5 是单参数版本：残差只约束 $\kappa a$，无内点数据时 $\kappa$ 与网络幅度沿 $\kappa a=1$ 滑动，所以低损失与错误参数可以同时出现。

**（2）边界条件满足不等于边界数据含信息。** 零边界对一切 $\kappa$ 成立，不能区分 $\kappa$；真正提供信息的是 $J_d=\partial u_\kappa/\partial\kappa$。若边界条件显含 $\kappa$（如 Robin 条件），边界数据会携带信息；不要把“边界观测无用”当成普适定律。

**（3）可辨识但病态。** E3 的观测点在内部，$H$ 单射，原则上是可辨识的；但 $s_d$ 小、Fisher 信息小，有限预算下估计方差大。E3 与 E4/E5 接近，说明“原则上唯一”不等于“实际能估准”。

**（4）评价网格与边界数据的细节。** 若评价网格使用 `linspace(0,1,1001)`，它会包含 E3 的 $0.01,\dots,0.05$ 和 E4/E5 的 $0,1$；直接用它做解 $L^2$ 会让评价点与观测点重合。本节主评价改用 1001 个单元中心点，避开观测点。E4/E5 的 $\mathcal L_b$ 与 $\mathcal L_d$ 也重合，解释 E5 时必须写清“已知边界 + 带噪边界测量”的竞争关系。

**（5）先验/正则化不是信息。** 先验能稳定数值、交换偏差与方差，但不能把统计不可辨识变成可辨识。若先验来自独立测量，它应作为额外数据纳入似然，而不是藏在正则项里；报告时要写清 $\hat\kappa$ 是数据主导还是先验主导。

### 随堂提问

**Q1（两个参数/状态组合解释相近观测）.** 设 $-\kappa u''=A\pi^2\sin(\pi x)$、零边界，$A,\kappa$ 都未知；取 $(\kappa,A)=(1,2)$ 与 $(2,4)$。证明两组给出相同解；只观测 $u$ 能否区分？若额外测得 $q_0=-\kappa u'(0)$，能否识别 $\kappa$？

**解释.** $u=A\sin(\pi x)/\kappa$，两组都给 $2\sin(\pi x)$，逐点相同；残差、边界、任意内点观测都相同，不能区分。这是射线退化，Fisher 矩阵秩 1。由 $q_0=-\kappa u'(0)=-A\pi$ 得 $\hat A=-q_0/\pi$；再由 $\hat r=\sum s_dy_d/\sum s_d^2$ 得 $\hat\kappa=\hat A/\hat r$。注意通量提供的是 $A$ 的信息，$\kappa$ 是靠“$A$ 已知 + 比值”间接得到的。

**Q2（噪声下能否区分两个很接近的 $\kappa$）.** $\kappa^\star=2$，布局 A（5 点，$x_d=d/6$），$\sigma=0.02$。$\kappa=2.05$ 与 $2$ 能否在 $95\%$ 水平区分？要分辨 $1\%$ 需要多少点？

**解释.** $S=\sqrt3$，$\mathrm{sd}(\hat\kappa)=0.02\cdot4/1.7321\approx0.0462$；差值 $0.05$ 的 $z\approx1.08<1.96$，不能区分；需 $\Delta\gtrsim0.091$。要分辨 $1\%$（$\Delta=0.02$），需 $\mathrm{sd}\le0.0102$，即 $S\ge7.84$、$\sum s_d^2\ge61.5$，均匀布点约需 $N_d\gtrsim123$。这是无偏、线性化下的界，PINN 实际只会更差。

**Q3（观测设计）.** 单参数 $\kappa$，3 个传感器，$\sigma$ 固定；怎么放使 $\sum\sin^2(\pi x_d)$ 最大？与等距 3 点比，CR 标准差差多少？若 $A$ 与 $\kappa$ 同时未知，设计能解决可辨识性吗？

**解释.** 逐点最大在 $x=1/2$；允许重合时三个都放 $1/2$ 得 3；互异时上确界仍是 3。等距 3 点的 $\sum s_d^2=2$，与上确界 3 相比 $\mathrm{sd}$ 之比 $\sqrt{3/2}\approx1.22$，差约 22%。若 $A,\kappa$ 同时未知，设计不能解决可辨识性：$\partial u/\partial A$ 与 $\partial u/\partial\kappa$ 成比例，Fisher 矩阵秩恒为 1，与位置无关；必须换观测类型或加独立信息。

**Q4（硬约束 + 仅边界数据）.** 用硬约束 $u_\theta=x(1-x)N_\theta$，只给零边界数据，损失降到 $10^{-10}$，能否说 $\kappa$ 已学会？给出两个诊断。

**解释.** 硬约束使 $\mathcal L_b\equiv0$；对任意 $\kappa$，$u_\kappa$ 都能写成 $x(1-x)N$，所以 $\mathcal L\equiv0$ 对一切 $\kappa$ 成立，$\partial\mathcal L/\partial\alpha\equiv0$。损失小只说明边界满足，不说明 $\kappa$ 定了。诊断一：多初值/多种子（$\kappa_0\in\{1,2,4\}$），若 $\hat\kappa$ 随初值系统移动而损失都近似 0，即不可辨识。诊断二：加一个内点观测再重训，若 $\hat\kappa$ 立刻跳到某个值且损失仍可很小，说明原先的平坦方向被数据钉住了。解析诊断：$I(\kappa)=0$ 或 $\sum_d\sin^2(\pi x_d)=0$。


### 6.2 实际运行记录

- **运行日期：** 2026-10-27。
- **运行环境：** Windows；Python 3.12.15（Anaconda）/ NumPy 2.5.4 / Matplotlib 3.11.2；CPU；未使用 scipy/torch。模板 kernel 名保留 `Python 3 (course-2026)` 仅作兼容标记，实际执行使用本机 Python 3.12.15。
- **运行范围：** 5 个种子（2026–2030）× 5 个设置 = 25 次完整运行；正文表 6.2-1、表 6.2-2、Notebook 输出和 `数据/` 中的逐次结果均来自本次运行。
- **代码状态：** `ch06-fill-011/012/013` 已实际执行；本 Notebook 末尾的“结果解释”和“6.2 实际运行记录”两个 Markdown 单元依据本次输出填写。训练代码为纯 NumPy 手写前向、解析二阶导和反向传播；Adam 2000 步、学习率 $0.01$；5 个种子 × 5 个设置 = 25 次。
- **梯度核验：** 有限差分检查通过：$u_{xx}$ 相对误差约 $7.24\times10^{-7}$，参数梯度最大相对误差约 $9.72\times10^{-8}$。这说明解析二阶导和反向传播的实现与数值差分一致。
- **运行预算：** 25 次训练总墙钟 96.9 s；多数单次 2.6–4.4 s，E5 的 seed 2027 因系统负载约 10.9 s，故 E5 时间均值 5.6 s。
- **配套说明：** 运行环境、汇总和数据文件说明见 `辅助文件/A1组-6.2-说明.md`。
- **数据位置：** `数据/A1组-6.2-反问题汇总.json`（全部指标）、`数据/A1组-6.2-反问题逐次结果.csv`（逐次表）、`数据/A1组-6.2-训练历史.npz`（损失/$\kappa$ 历史）、`数据/A1组-6.2-预测数据.npz`（评价网格预测）、`图片/A1组-6.2-反问题主图.png`、`图片/A1组-6.2-E5边界噪声收敛.png`、`图片/A1组-6.2-E1解对比.png`。
- **实际结果（Python 3.12.15，5 seeds）：** E1 参数误差 6.89%±0.93%、解 $L^2$ 5.14%±1.83%；E2 6.33%±0.64%、4.03%±0.31%；E3 31.69%±3.05%、44.82%±6.39%；E4 33.02%±2.33%、48.86%±5.62%；E5 33.12%±2.38%、50.11%±5.03%。逐次结果见 `ch06-fill-013` 输出；没有只展示最好一次。
- **结果口径：** 本节只报告本次 Python 3.12.15 复现的 25 次结果；参数误差、解 $L^2$、观测 RMSE、残差 RMS、边界误差和运行时间均来自 `数据/A1组-6.2-反问题逐次结果.csv`。
- **异常与说明：** E1 的 seed 2027 出现残差 RMS 0.4036、边界误差 0.0590，E2 的 seed 2028 出现残差 RMS 0.1931、边界误差 0.0398，E4 的 seed 2030 出现边界误差 0.0624；这些是单次优化的尾部结果，已保留在逐次表中，没有删除。E4/E5 的 $\mathcal L_b$ 与 $\mathcal L_d$ 在边界点重合，带噪边界数据与已知零边界会竞争；E5 的观测 RMSE/边界误差更小不代表参数更准。
- **结论：** 内点观测（E1/E2）明显优于近边界（E3）和仅边界（E4/E5）；E4/E5 出现“损失低、参数错”的平坦行为；多种子散度不可忽略。该结论只适用于一维、线性、常数 $\kappa$、零边界、合成观测。
- **尚未完成/未核验：** 课程 `course-2026` kernel 未在本机注册，不影响 Python 3.12.15 运行；硬约束、先验/正则化未进入主实验；噪声未扩展到第二个非零水平；报告模板的“被审内容版本”仍需完整 commit SHA；B/C 复测尚未进行。上述项不写成已完成。


### 本 Notebook 产物

| 文件 | 内容 |
| --- | --- |
| `数据/A1组-6.2-反问题汇总.json` | 25 次运行的全部指标（JSON） |
| `数据/A1组-6.2-反问题逐次结果.csv` | 25 次逐次结果表（CSV） |
| `数据/A1组-6.2-训练历史.npz` | 每次训练的损失与参数历史 |
| `数据/A1组-6.2-预测数据.npz` | 评价网格上的预测解 |
| `图片/A1组-6.2-反问题主图.png` | E1/E4 收敛、灵敏度、参数误差分布 |
| `图片/A1组-6.2-E5边界噪声收敛.png` | E5 五个种子的参数收敛 |
| `图片/A1组-6.2-E1解对比.png` | E1 五个种子的解与逐点误差 |

> 说明：代码只写入 `数据/` 与 `图片/` 两个既有目录，不创建新文件夹；正文中的 `ch06-fill-011/012/013` 即本 Notebook 的三个代码单元 ID。
